In [1]:
import os
import sys
sys.path.append(os.path.abspath('../scripts'))
fig_path      = '../figures/'
data_path     = '../data/'

In [2]:
import netCDF4 as nc
import numpy as np
import xarray as xr
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd 
import seaborn as sns
import cmocean as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
from proj_utils import *
from mapping_utils import *
from plotting_utils import *

In [ ]:
window_sz = 48
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for c in range(len(ds_slp.time)-(window_sz)):
    #for c in range(10):
        # --- Slice SLP ---
        slp_temp = ds_slp['msl'].sel(time = slice(ds_slp['msl'].time[c].data, ds_slp['msl'].time[c+window_sz].data))

        # --- Slice WSC --- 
        wsc_temp = ds_wsc['wsc'].sel(time = slice(ds_wsc['wsc'].time[c].data, ds_wsc['wsc'].time[c+window_sz].data))
        
        # --- Calculate modes ---
        eofs_temp, pcs_temp, per_var_temp, eigs_temp = calc_eofs(slp_temp, num_modes=4)
        mode_one_temp = eofs_temp[0,:,:]
        mode_two_temp = eofs_temp[1,:,:]

        pcs_unscaled = pcs_temp*np.sqrt(eigs_temp[:4])
        mode_one_unscaled = pcs_unscaled[:,0]
        mode_two_unscaled = pcs_unscaled[:,1]
        
        # --- Correct the sign of the modes ---
        corr_nao_temp = xr.corr(nao_sp,eofs_temp[0,:,:])
        corr_eap_temp = xr.corr(nao_sp,eofs_temp[1,:,:])
        
        if abs(corr_nao_temp) > abs(corr_eap_temp):
            sign_nao = xr.corr(nao_sp,eofs_temp[0,:,:])
            sign_eap = xr.corr(eap_sp,eofs_temp[1,:,:])
            if sign_nao < 0:
                mode_one_unscaled = - mode_one_unscaled
            if sign_eap < 0 :
                mode_two_unscaled = - mode_two_unscaled
            reg_nao_wsc   = linregress_spatial(mode_one_unscaled, wsc_temp)
            reg_eap_wsc   = linregress_spatial(mode_two_unscaled, wsc_temp)
        else:
            sign_nao = xr.corr(nao_sp,eofs_temp[1,:,:])
            sign_eap = xr.corr(eap_sp,eofs_temp[0,:,:])
            if sign_nao < 0:
                mode_two_unscaled = - mode_two_unscaled
            if sign_eap < 0 :
                mode_one_unscaled = - mode_one_unscaled
            reg_nao_wsc   = linregress_spatial(mode_two_unscaled, wsc_temp)
            reg_eap_wsc   = linregress_spatial(mode_one_unscaled, wsc_temp)
